# Example 11.5 / 1: Standard Cone

Companion notebook for **11.5 Second-Order Cones** in the AMPL lab Documentation, section "Standard Cone". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Standard Cone

The squared form `sum {i in I} x[i]^2 <= t^2` is equivalent only when `t >= 0` is also enforced.

On the point $x=(3,4)$, where $\lVert x\rVert_2=5$:

| Value of `t` | `sqrt(sum x^2) <= t` | `sum x^2 <= t^2` |
|---|---|---|
| `5` | holds: $5\le5$ | holds: $25\le25$ |
| `4` | fails: $5>4$ | fails: $25>16$ |
| `-5` | fails: $5>-5$ | holds: $25\le25$, so the squared form admits a negative `t` |

Minimizing `t` with $x$ fixed at $(3,4)$ makes the difference observable. With the squared form
alone Ipopt drives `t` far below $-5$; adding `t >= 0` restores the cone answer $t=5$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x {1..2};  # cone coordinates
var t;         # radius; no sign restriction here

minimize T: t;  # smallest radius

subject to Fix1: x[1] = 3;                   # first coordinate of the point (3, 4)
subject to Fix2: x[2] = 4;                   # second coordinate of the point (3, 4)
subject to Squared: x[1]^2 + x[2]^2 <= t^2;  # squared form: also allows t <= -5

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.var["t"].set_value(-1)
ampl.solve(solver="ipopt", verbose=False)
print("squared form only:", ampl.solve_result, ampl.var["t"].value() < -1000)
ampl.eval("subject to Sign: t >= 0;  # the radius sign the squared form lacks")
ampl.var["t"].set_value(1)
ampl.solve(solver="ipopt", verbose=False)
print("with t >= 0:", ampl.solve_result, round(ampl.var["t"].value(), 6))

squared form only: solved True
with t >= 0: solved 5.0
